# Customer Churn Prediction System
### Phase 4: Train Models (Task 11)

This phase trains four different models on the exact same prepared data from
Phase 3, then generates predictions from each on the test set. **We are not
scoring or comparing them yet** — that's Phase 5. Right now the only goal is:
get four trained models, each producing predictions.

## Setup: rebuilding Phase 3's prepared data

Since each notebook stands on its own, this quickly re-runs Phase 3's steps
(clean, split, encode, scale) to get back to `X_train_prep` / `X_test_prep` /
`y_train` / `y_test`. Nothing new here — see `03_prepare_for_ml.ipynb` for the
full explanation of every line.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('customer_churn.csv').drop(columns=['CustomerID'])
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

gender_map = {'Male': 0, 'Female': 1}
X_train_prep['Gender'] = X_train_prep['Gender'].map(gender_map)
X_test_prep['Gender'] = X_test_prep['Gender'].map(gender_map)

categorical_cols = ['Subscription Type', 'Contract Length']
X_train_prep = pd.get_dummies(X_train_prep, columns=categorical_cols, drop_first=True)
X_test_prep = pd.get_dummies(X_test_prep, columns=categorical_cols, drop_first=True)
X_test_prep = X_test_prep.reindex(columns=X_train_prep.columns, fill_value=0)

numeric_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction']
scaler = StandardScaler()
X_train_prep[numeric_cols] = scaler.fit_transform(X_train_prep[numeric_cols])
X_test_prep[numeric_cols] = scaler.transform(X_test_prep[numeric_cols])

print(f"Ready. X_train_prep: {X_train_prep.shape}   X_test_prep: {X_test_prep.shape}")

Ready. X_train_prep: (51499, 12)   X_test_prep: (12875, 12)


## Task 11: Train Four Models

Every model below follows the exact same two-step pattern:
1. **`.fit(X_train_prep, y_train)`** — the model studies the training data and
   learns the patterns connecting features to churn.
2. **`.predict(X_test_prep)`** — the trained model looks at the 12,875 unseen
   test customers and guesses churn (0 or 1) for each one.

We'll do this once per model, and keep every model's predictions so Phase 5 can
score them all fairly.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Store trained models and their predictions here, one entry per model
predictions = {}

### Model 1: Logistic Regression

The simplest of the four. It learns a weighted combination of all 12 features
and uses that to estimate the probability of churn — if the weighted sum passes
a threshold, it predicts churn.

In [3]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_prep, y_train)
predictions['Logistic Regression'] = log_reg.predict(X_test_prep)

print("Logistic Regression trained.")
print(f"Predicted churn count: {predictions['Logistic Regression'].sum()} out of {len(y_test)}")

Logistic Regression trained.
Predicted churn count: 6171 out of 12875


### Model 2: Random Forest

Builds many decision trees, each trained on a slightly different random slice of
the data, then lets them vote. Trees naturally handle the sharp thresholds we
found in Phase 2 well (e.g. "Support Calls > 4?").

In [4]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_prep, y_train)
predictions['Random Forest'] = rf.predict(X_test_prep)

print("Random Forest trained.")
print(f"Predicted churn count: {predictions['Random Forest'].sum()} out of {len(y_test)}")

Random Forest trained.
Predicted churn count: 6084 out of 12875


### Model 3: SVM (Support Vector Machine)

Finds the boundary that separates churners from non-churners with the widest
possible margin between the two groups, rather than just any boundary that
happens to work.

In [5]:
svm = SVC(random_state=42)
svm.fit(X_train_prep, y_train)
predictions['SVM'] = svm.predict(X_test_prep)

print("SVM trained.")
print(f"Predicted churn count: {predictions['SVM'].sum()} out of {len(y_test)}")

SVM trained.
Predicted churn count: 6292 out of 12875


### Model 4: K-Nearest Neighbors (KNN)

For each test customer, KNN finds the 5 most similar customers in the training
data (by feature values) and predicts churn based on what the majority of those
5 neighbors did. `n_neighbors=5` is a common default starting point.

In [6]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_prep, y_train)
predictions['KNN'] = knn.predict(X_test_prep)

print("KNN trained.")
print(f"Predicted churn count: {predictions['KNN'].sum()} out of {len(y_test)}")

KNN trained.
Predicted churn count: 6513 out of 12875


## Quick sanity check (not scoring yet)

Just to see all four side by side against reality, here are the first 10 test
customers: the actual outcome, and what each of the four models guessed.

In [7]:
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Logistic Regression': predictions['Logistic Regression'][:10],
    'Random Forest': predictions['Random Forest'][:10],
    'SVM': predictions['SVM'][:10],
    'KNN': predictions['KNN'][:10],
})
comparison

,Actual,Logistic Regression,Random Forest,SVM,KNN
0,0,0,0,0,0
1,1,0,1,1,1
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0
5,0,0,0,0,0
6,0,0,0,0,0
7,0,0,0,0,0
8,0,0,0,0,0
9,0,0,0,0,0


**Notice how the predicted churn counts differ across models** — that's
expected, and it's *exactly* why we score them properly in Phase 5 instead of
guessing which one is best from a glance. A model that predicts churn for almost
everyone might look "aggressive" here but could actually have poor precision;
one that predicts churn rarely might miss real churners. Raw counts don't tell
that story — proper metrics do.

---
### ✅ Phase 4 checkpoint

You now have four trained models — `log_reg`, `rf`, `svm`, `knn` — and a
`predictions` dictionary holding each one's guesses on the same 12,875 test
customers.

**Next: Phase 5 (Tasks 12-16)** — properly scoring all four with accuracy,
precision, recall, and F1, building a confusion matrix for the best one, checking
feature importance, and then tuning that best model.